In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


transform=transforms.Compose([
    transforms.ToTensor(),#tensordataset and scaling together
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))  #mean,standard deviation 
])

trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

In [7]:
trainLoader=DataLoader(trainset,batch_size=64,shuffle=True)
testLoader=DataLoader(testset,batch_size=64)

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size=3,padding=1),#use output formula
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        
        )  

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10),
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)#falttening
        x=self.fc_layers(x)

        return x
        

In [10]:
model=CNN()

In [12]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainLoader:
        optimizer.zero_grad()

        output=model.forward(images)
        loss=criteria(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainLoader)}")  


In [ ]:
#evaluate

correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images.labels in testLoader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)

print(f"accuracy={correct_labels/total_labels*100")